In [1]:
import math
import os
import hydra
import tonic
from tonic.datasets import CIFAR10DVS
import torch
from tonic.transforms import Optional, ToFrame
from datasets.utils.pad_tensors import PadTensors
from datasets.utils.diskcache import DiskCachedDataset
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split

/raid/home/michael.siegl/projects/SE-adlif/.venv/lib/python3.11/site-packages/lightning_fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


## Data Preparation
We first need to preprocess the raw data found in the data path into a **Tonic** dataset. 

In [13]:
data_path = "/home/michael/projects/welding-data/raw"
# identify files, ending either in .raw or .bias
raw_files = []
bias_files = []
# check path
print(f"Checking path: {data_path}")
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Path {data_path} does not exist.")

raw_files, bias_files

Checking path: /home/michael/projects/welding-data/raw


FileNotFoundError: Path /home/michael/projects/welding-data/raw does not exist.

In [ ]:
class WeldingArm(tonic.datasets.Dataset):
    def __init__(self, root=data_path, transform=None, target_transform=None):
        super().__init__(root, transform, target_transform)
        self.data = DiskCachedDataset(os.path.join(root, "welding_arm"))
        self.classes = ["arm_up", "arm_down", "arm_left", "arm_right"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        events = sample["events"]
        label = sample["label"]
        if self.transform:
            events = self.transform(events)
        if self.target_transform:
            label = self.target_transform(label)
        return events, label